# Orbitstack NRR Decline — Commercial Diagnostic

Synthetic data demonstration of a 2-week commercial analytics diagnostic
for a fictional PE-backed B2B SaaS business. All figures are generated;
no real client data is present.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

DATA_DIR = Path("data")
FIGURES_DIR = Path("outputs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = pd.Timestamp("2025-12-31")
ANALYSIS_START = pd.Timestamp("2023-01-01")
ANALYSIS_END = pd.Timestamp("2025-12-31")

PALETTE = {
    "primary":    "#1F3A5F",
    "secondary":  "#4A7C9D",
    "accent":     "#C8553D",
    "neutral":    "#6B7280",
    "positive":   "#4A7C59",
    "background": "#F8F9FA",
    "text":       "#1F2937",
}

SEGMENT_COLOURS = {
    "SMB":         "#6B9BD1",
    "Mid-market":  "#C8553D",
    "Enterprise":  "#1F3A5F",
}

SEGMENT_LINESTYLES = {
    "SMB":         "--",
    "Mid-market":  "-",
    "Enterprise":  "-.",
}

MOVEMENT_COLOURS = {
    "starting":    "#6B7280",
    "new":         "#4A7C59",
    "expansion":   "#4A7C9D",
    "contraction": "#C8553D",
    "churn":       "#8B3A2F",
    "ending":      "#1F3A5F",
}

SEGMENT_ORDER = ["SMB", "Mid-market", "Enterprise"]

SOURCE_ANNOTATION = (
    "Source: Orbitstack subscription data, Q1 2022 – Q4 2025. "
    "Synthetic data for demonstration purposes."
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor":   PALETTE["background"],
    "axes.edgecolor":   PALETTE["neutral"],
    "axes.labelcolor":  PALETTE["text"],
    "text.color":       PALETTE["text"],
    "xtick.color":      PALETTE["text"],
    "ytick.color":      PALETTE["text"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi":        100,
})


def add_source_annotation(fig: plt.Figure, text: str = SOURCE_ANNOTATION) -> None:
    fig.text(
        0.01, -0.02, text,
        fontsize=8, color=PALETTE["neutral"], ha="left", va="top",
    )


## Section 0 — Glossary

Short reference list. Definitions expanded in section 3.

- **ARR** — Annual Recurring Revenue. Contracted subscription revenue for the
  forward twelve months, measured on active paying customers.
- **CARR** — Committed ARR including signed-but-not-yet-billing deals. Not
  used in this notebook; only active ARR is in scope.
- **NRR** — Net Revenue Retention. Ending ARR of a cohort divided by that
  cohort's starting ARR, including expansion and contraction but excluding
  new logos acquired in the analysis period.
- **GRR** — Gross Revenue Retention. NRR without the expansion
  contribution — a pure retention lens.
- **Churn** — used here in the revenue sense (lost ARR) unless prefixed
  "logo". Logo churn counts customers; revenue churn weights by ARR.
- **Expansion / Contraction** — upward and downward moves in ARR while a
  customer remains active. Full departure is churn, not contraction.
- **Cohort** — the calendar quarter of a customer's first paid invoice
  date. Used as the retention anchor throughout.


## Section 2 — Engagement context

> **Orbitstack** is a PE-backed B2B SaaS business providing workflow
> automation to mid-market logistics firms. Mid-twenties-millions ARR as of
> 31 December 2025, 3 years into a 5-year hold period with sponsor
> **Meridian Capital**. The sponsor has requested a commercial analytics
> review ahead of a planned exit process in 18–24 months.
>
> **The trigger:** Headline NRR has declined from 118% in Q1 2023 to 104%
> in Q4 2025. The CFO's initial view is that this is a churn problem driven
> by macro pressure on logistics customers. The sponsor wants an
> independent diagnostic.
>
> **This notebook represents the first two weeks of the engagement** —
> data collection, definitional audit, decomposition, segmentation, and
> initial hypothesis formation.

The questions this notebook answers: Is the NRR decline real under a
defensible definition? Is it driven by churn or by expansion collapse? Is
it concentrated in a segment? Do the data support a competitive or
product-led explanation, or is the cause elsewhere? What would the
priority workstreams be for weeks 3–8?


## Section 3 — Definitional audit

Before reproducing the trigger metric we fix the definitions. The sponsor
and the CFO cite a headline NRR number; we need to know exactly which NRR
that is and where the denominators sit.

**NRR definition.** We use dollar-based, trailing-twelve-months Net
Revenue Retention, expressed as a percentage. For every reporting
quarter, the numerator is the ARR at quarter end from customers who were
active twelve months earlier; the denominator is the ARR those same
customers contributed at the start of that twelve-month window. New
logos acquired inside the analysis window (Q1 2023 – Q4 2025) are
excluded from both sides so the metric measures retention rather than
blended retention-plus-growth. New-logo ARR is reported separately in the
decomposition in section 6.

**Cohort anchor.** Customer cohort is the calendar quarter of
`first_paid_invoice_date` — the billing system record. The CRM
`acquisition_date` lags by 0–3 months in the ordinary course and by up
to 90 days for a small number of customers on free-trial gating (flagged
as DQ quirk C in section 4). Anchoring to billing ensures retention curves
rest on the date cash actually starts rather than on sales-stage
timestamps.

**Scope exclusions.** Re-activations are excluded — no churned customer
returns in the generated dataset, and production data would need
dedicated handling before this metric could absorb them. CARR and backlog
are out of scope; active ARR only is used throughout. All values are
GBP-denominated with no FX translation applied; the business is
UK-domiciled and EU customers are billed in GBP.

**Timebase.** The snapshot is 31 December 2025. No relative dates appear
in analysis or commentary. Quarters are calendar quarters. The generator
runs from Q1 2021 so the Q1 2023 TTM denominator rests on a fully
established 2022 opening book.

### Assumptions requiring client finance-team validation

1. `first_paid_invoice_date` is the canonical cohort anchor; five
   customers with a CRM-to-billing gap exceeding 30 days are treated as
   free-trial overlaps rather than data errors. Confirm with the CS /
   finance workflow owner.
2. Customers flagged `Churned` on their final subscription row are
   assumed to have ceased paying on that date. Any live re-activation
   pattern would need an explicit treatment.
3. Effective price per seat is taken verbatim from the billing export;
   discount-authorisation trail (rep, deal, approver) is not
   reconstructed from the CSVs in this diagnostic.
4. The four-quarter rolling average used for segment NRR in section 7
   consumes Q2–Q4 2022 as warmup input. Those pre-period quarters are
   present in the generated data and are assumed to be analytically
   comparable to the Q1 2023 opening book.


## Section 4 — Data overview, provenance, and quality

Three CSVs are produced by `data/generate_data.py` with a fixed seed of
42. Row counts, date ranges, segment split, provenance, lineage, and data
quality quirks are tabulated below before any analytical work begins.


In [ ]:
customers = pd.read_csv(
    DATA_DIR / "customers.csv",
    parse_dates=["acquisition_date", "first_paid_invoice_date"],
)
subscriptions = pd.read_csv(
    DATA_DIR / "subscriptions.csv",
    parse_dates=["month_end_date", "contract_end_date"],
)
churn_reasons = pd.read_csv(
    DATA_DIR / "churn_reasons.csv",
    parse_dates=["churn_date"],
)

customers["segment"] = pd.Categorical(
    customers["segment"], categories=SEGMENT_ORDER, ordered=True
)


### Row counts and date ranges


In [ ]:
row_counts = pd.DataFrame(
    [
        {
            "file": "customers.csv",
            "rows": len(customers),
            "date_column": "first_paid_invoice_date",
            "min_date": customers["first_paid_invoice_date"].min().date(),
            "max_date": customers["first_paid_invoice_date"].max().date(),
        },
        {
            "file": "subscriptions.csv",
            "rows": len(subscriptions),
            "date_column": "month_end_date",
            "min_date": subscriptions["month_end_date"].min().date(),
            "max_date": subscriptions["month_end_date"].max().date(),
        },
        {
            "file": "churn_reasons.csv",
            "rows": len(churn_reasons),
            "date_column": "churn_date",
            "min_date": churn_reasons["churn_date"].min().date(),
            "max_date": churn_reasons["churn_date"].max().date(),
        },
    ]
)
row_counts


### Segment split (customer count and share of original ARR)


In [ ]:
segment_split = (
    customers.groupby("segment", observed=True)
    .agg(customers=("customer_id", "size"),
         original_arr_gbp=("original_arr_gbp", "sum"))
)
segment_split["customer_share"] = (
    segment_split["customers"] / segment_split["customers"].sum()
)
segment_split["arr_share"] = (
    segment_split["original_arr_gbp"] / segment_split["original_arr_gbp"].sum()
)
segment_split["original_arr_gbp"] = segment_split["original_arr_gbp"].map(
    lambda v: f"£{v/1_000_000:.2f}m"
)
segment_split["customer_share"] = segment_split["customer_share"].map(
    lambda v: f"{v*100:.1f}%"
)
segment_split["arr_share"] = segment_split["arr_share"].map(
    lambda v: f"{v*100:.1f}%"
)
segment_split


### Data provenance

Each CSV corresponds to a fictional source system owned by a different
team. The mapping is narrative context for the diagnostic; the CSVs
themselves are standardised extracts and do not carry source-system
specific fields.

| File | Fictional source | Owning team |
|---|---|---|
| `customers.csv` | CRM export | Revenue Operations |
| `subscriptions.csv` | Billing system export | Finance |
| `churn_reasons.csv` | Customer success platform export | Customer Success |


### Data lineage

```
customers.csv         ─┬─→ section 4 (overview, provenance, DQ)
                       │   → section 7 (segmentation by segment)
                       │   → section 8 (cohort = first_paid_invoice_date quarter)
                       └─→ section 10 (acquisition-era facet)

subscriptions.csv     ─┬─→ section 5 (headline NRR)
                       │   → section 6 (NRR decomposition + reconciliation)
                       │   → section 6a (sensitivity variants)
                       │   → section 7 (quarterly NRR by segment)
                       │   → section 8 (retained ARR by cohort)
                       └─→ section 10 (seat expansion by cohort)

churn_reasons.csv     ───→ section 9 (churn reason taxonomy, FleetLogic gate)
```

Upstream transformations applied before the CSV boundary (customer
deduplication, invoice → ARR roll-up, churn attribution cleanup) are
owned by the respective source teams and are treated as trusted inputs
here.


### Data quality quirks

Three deliberate quirks are present in the generated data. Each mirrors a
reconciliation pattern common in commercial due-diligence work. Handling
decisions are recorded so they can be re-applied in production pipelines.

| Quirk | Detection | Handling | Impact on findings |
|---|---|---|---|
| A — Billing system £1 dips. Three customers show a single-month ARR dip of exactly £1 mid-history. | Vectorised check for rows where `arr_gbp - prev_arr == -1.0` and `arr_gbp - next_arr == -1.0` while `subscription_status == 'Active'`. | Forward-fill the affected month with prior-month ARR before computing section 6 decomposition movements. | Negligible — affects three customer-months out of 16,000+. |
| B — Backdated ARR adjustment. One customer has an unrecorded mid-contract uplift reconciled in month 6. | Same vectorised rule with sign flipped (`+£500` spike against prior and next months). | Absorb the uplift into the baseline ARR from month 5 onwards (treat the month-6 spike as calendar noise, not expansion). | Negligible — affects one customer-month. |
| C — CRM vs billing definitional mismatch. Five customers have `acquisition_date` preceding `first_paid_invoice_date` by 30–90 days (free-trial period not reflected in the CRM date). | `first_paid_invoice_date - acquisition_date` exceeds 30 days. Because 0–90 day gaps occur in the baseline population too, the five cannot be uniquely identified by gap size alone. | Use `first_paid_invoice_date` as the cohort anchor throughout. Document the gap as a class of issue rather than naming specific IDs. Requires finance-team confirmation of trial-billing workflow. | Minor — shifts five cohorts by one quarter at most; TTM NRR windows are wide enough to absorb the shift. |


### Customer acquisition by quarter

One overview chart closes section 4: a stacked bar of customer count by
segment by acquisition quarter. It establishes that the book is heavily
front-loaded in 2021 (a deliberate feature of the baseline population,
noted in section 3) and that recent intake has tilted toward smaller
segments.


In [ ]:
in_book = customers[customers["first_paid_invoice_date"] <= SNAPSHOT_DATE].copy()
post_snapshot_count = len(customers) - len(in_book)

acq_quarter = in_book["first_paid_invoice_date"].dt.to_period("Q")
counts_by_quarter = (
    in_book.assign(acq_quarter=acq_quarter)
    .pivot_table(
        index="acq_quarter",
        columns="segment",
        values="customer_id",
        aggfunc="count",
        fill_value=0,
        observed=True,
    )
    .reindex(columns=SEGMENT_ORDER, fill_value=0)
    .sort_index()
)
counts_by_quarter.index = counts_by_quarter.index.astype(str)

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(counts_by_quarter))
x = np.arange(len(counts_by_quarter))
for segment in SEGMENT_ORDER:
    values = counts_by_quarter[segment].to_numpy()
    ax.bar(
        x, values, bottom=bottom,
        color=SEGMENT_COLOURS[segment],
        edgecolor=PALETTE["background"], linewidth=0.5,
        label=segment,
    )
    bottom = bottom + values

year_2021_total = int(counts_by_quarter.loc[["2021Q1", "2021Q2", "2021Q3", "2021Q4"]].to_numpy().sum())
book_total = int(counts_by_quarter.to_numpy().sum())
book_share_2021 = year_2021_total / book_total

ax.set_xticks(x)
ax.set_xticklabels(counts_by_quarter.index, rotation=45, ha="right")
ax.set_xlabel("Acquisition quarter (first paid invoice)")
ax.set_ylabel("Customers acquired (count)")
ax.set_title(
    f"Customer intake front-loaded: {book_share_2021*100:.0f}% of the active book landed in 2021; "
    "mid-market intake thins from 2024",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(title="Segment", loc="upper right", frameon=False)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

footnote = (
    f"Includes the {book_total} customers with a first paid invoice on or before "
    f"31 December 2025. A further {post_snapshot_count} customers were acquired late in 2025 "
    f"and had not yet billed at the snapshot."
)
fig.text(0.01, -0.02, footnote, fontsize=8, color=PALETTE["neutral"], ha="left", va="top")
fig.text(0.01, -0.06, SOURCE_ANNOTATION, fontsize=8, color=PALETTE["neutral"], ha="left", va="top")

fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "04_acquisition_by_segment.png",
    dpi=150, bbox_inches="tight",
)
plt.show()


## Section 5 — Headline NRR reproduction

Reproduces the trigger metric from the engagement context using the
definition agreed in section 3: trailing-twelve-months, dollar-based
Net Revenue Retention, excluding customers acquired during the
analysis period. The TTM cohort at quarter `Q` is the set of
customers with positive ARR at quarter end `Q-4`; NRR for `Q` is the
ratio of those customers' ARR at `Q` to their ARR at `Q-4`.

In [ ]:
new_logo_ids = customers.loc[
    customers["first_paid_invoice_date"] >= ANALYSIS_START, "customer_id"
]

WARMUP_START = ANALYSIS_START - pd.DateOffset(months=12)
all_quarter_ends = pd.date_range(WARMUP_START, ANALYSIS_END, freq="QE")
analysis_quarter_ends = pd.date_range(ANALYSIS_START, ANALYSIS_END, freq="QE")

active_quarter_arr = subscriptions[
    (subscriptions["month_end_date"].isin(all_quarter_ends))
    & (subscriptions["subscription_status"] == "Active")
    & (~subscriptions["customer_id"].isin(new_logo_ids))
]
arr_pivot = active_quarter_arr.pivot_table(
    index="customer_id",
    columns="month_end_date",
    values="arr_gbp",
    aggfunc="sum",
    fill_value=0.0,
)

ttm_start_quarter_ends = pd.DatetimeIndex(
    analysis_quarter_ends - pd.DateOffset(months=12)
)
start_arr_matrix = arr_pivot.reindex(columns=ttm_start_quarter_ends, fill_value=0.0)
end_arr_matrix = arr_pivot.reindex(columns=analysis_quarter_ends, fill_value=0.0)

cohort_mask = start_arr_matrix.to_numpy() > 0
starting_total = (start_arr_matrix.to_numpy() * cohort_mask).sum(axis=0)
ending_total = (end_arr_matrix.to_numpy() * cohort_mask).sum(axis=0)

nrr_quarterly = pd.DataFrame({
    "quarter": analysis_quarter_ends.to_period("Q").astype(str),
    "starting_arr_gbp": starting_total,
    "ending_arr_gbp": ending_total,
    "nrr": ending_total / starting_total,
})

nrr_quarterly.assign(
    starting_arr_gbp=lambda d: d["starting_arr_gbp"].map(lambda v: f"£{v/1_000_000:.2f}m"),
    ending_arr_gbp=lambda d: d["ending_arr_gbp"].map(lambda v: f"£{v/1_000_000:.2f}m"),
    nrr=lambda d: (d["nrr"] * 100).map(lambda v: f"{v:.1f}%"),
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.axhline(
    1.0, linestyle=":", color=PALETTE["neutral"], linewidth=1.0,
    label="100% reference",
)
ax.plot(
    nrr_quarterly["quarter"],
    nrr_quarterly["nrr"],
    marker="o", linewidth=2.2, markersize=7,
    color=PALETTE["primary"],
    label="Trailing-12-months NRR",
)

first_row = nrr_quarterly.iloc[0]
last_row = nrr_quarterly.iloc[-1]
ax.annotate(
    f"Q1 2023: {first_row['nrr']*100:.1f}%",
    xy=(0, first_row["nrr"]),
    xytext=(12, 14), textcoords="offset points",
    color=PALETTE["primary"], fontsize=10, fontweight="bold",
    arrowprops=dict(arrowstyle="-", color=PALETTE["primary"], lw=0.6),
)
ax.annotate(
    f"Q4 2025: {last_row['nrr']*100:.1f}%",
    xy=(len(nrr_quarterly) - 1, last_row["nrr"]),
    xytext=(-110, -30), textcoords="offset points",
    color=PALETTE["accent"], fontsize=10, fontweight="bold",
    arrowprops=dict(arrowstyle="-", color=PALETTE["accent"], lw=0.6),
)

ax.set_xlabel("Quarter")
ax.set_ylabel("Net Revenue Retention (%)")
ax.set_xticks(range(len(nrr_quarterly)))
ax.set_xticklabels(nrr_quarterly["quarter"], rotation=45, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
y_lo = float(nrr_quarterly["nrr"].min()) - 0.04
y_hi = float(nrr_quarterly["nrr"].max()) + 0.04
ax.set_ylim(min(0.98, y_lo), max(1.22, y_hi))

start_pct = first_row["nrr"] * 100
end_pct = last_row["nrr"] * 100
ax.set_title(
    f"Net Revenue Retention declines from {start_pct:.0f}% to {end_pct:.0f}% across 12 quarters",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(loc="lower left", frameon=False)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "05_headline_nrr_decline.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

The TTM dollar-based NRR drops from 115.6% in Q1 2023 to 104.9% in
Q4 2025 — an 11 percentage point decline that reproduces the trigger
described in the engagement context. The trajectory is not monotonic:
NRR climbs above 120% through Q2 2024 before falling sharply across
the next four quarters. Whether that drop is driven by churn or by
expansion collapse — the CFO's working hypothesis versus the
alternative — is the question section 6 answers by decomposing each
quarter's ARR movements.

## Section 6 — NRR decomposition

Section 5 reproduced the headline decline but did not decompose it. The
12 quarterly TTM NRR values conceal four distinct movements — new ARR,
expansion, contraction, and churn — that together drive the net
change. This section reconstructs those movements quarter by quarter
for every customer, asserts that they tie back to the observed ending
ARR (no rounding drift, no missing rows), and then compares NRR against
its churn-only sibling GRR to separate the expansion story from the
retention story.

The reconciliation uses a full customer × quarter ARR pivot. For each
analysis quarter Q, a retention cohort is defined as customers with
positive ARR at Q-1. Flows are classified vectorised across the pivot:
new ARR for customers who first billed in Q, expansion and contraction
for in-cohort ARR moves, and churn for in-cohort customers who dropped
to zero ARR. The reconciliation identity is
`ending = starting + new + expansion - contraction - churn` and is
enforced by assertion before any chart is drawn.

In [ ]:
# Build one QE-spaced index and slice paired start/end views rather
# than subtracting DateOffset(months=3) — the latter drifts by a day
# on mid-quarter dates (Jun 30 − 3M = Mar 30, missing the Mar 31 QE).
decomp_all_ends = pd.date_range(
    ANALYSIS_START - pd.DateOffset(months=3), ANALYSIS_END, freq="QE",
)
decomp_end_dates = decomp_all_ends[1:]
decomp_start_dates = decomp_all_ends[:-1]

arr_quarter_pivot = (
    subscriptions[
        (subscriptions["month_end_date"].isin(decomp_all_ends))
        & (subscriptions["subscription_status"] == "Active")
    ]
    .pivot_table(
        index="customer_id",
        columns="month_end_date",
        values="arr_gbp",
        aggfunc="sum",
        fill_value=0.0,
    )
    .reindex(columns=decomp_all_ends, fill_value=0.0)
)

end_arr_q = arr_quarter_pivot.reindex(columns=decomp_end_dates, fill_value=0.0).to_numpy()
start_arr_q = arr_quarter_pivot.reindex(columns=decomp_start_dates, fill_value=0.0).to_numpy()

retention_cohort = start_arr_q > 0
new_logo_this_q = (start_arr_q == 0) & (end_arr_q > 0)
diff_arr = end_arr_q - start_arr_q

starting_total_q    = (start_arr_q * retention_cohort).sum(axis=0)
ending_total_q      = end_arr_q.sum(axis=0)
new_arr_q           = (end_arr_q * new_logo_this_q).sum(axis=0)
expansion_arr_q     = np.where(retention_cohort & (end_arr_q > 0) & (diff_arr > 0),  diff_arr, 0.0).sum(axis=0)
contraction_arr_q   = np.where(retention_cohort & (end_arr_q > 0) & (diff_arr < 0), -diff_arr, 0.0).sum(axis=0)
churn_arr_q         = np.where(retention_cohort & (end_arr_q == 0), start_arr_q, 0.0).sum(axis=0)

reconciliation_gap = ending_total_q - (
    starting_total_q + new_arr_q + expansion_arr_q - contraction_arr_q - churn_arr_q
)
max_gap = float(np.max(np.abs(reconciliation_gap)))
if max_gap > 0.01:
    worst_idx = int(np.argmax(np.abs(reconciliation_gap)))
    worst_quarter = decomp_end_dates[worst_idx].to_period("Q")
    raise AssertionError(
        f"ARR decomposition mismatch at {worst_quarter}: "
        f"gap £{reconciliation_gap[worst_idx]:,.2f}"
    )
print("Decomposition reconciled — all 12 quarters tie")

nrr_quarterly_decomp = (
    starting_total_q - churn_arr_q - contraction_arr_q + expansion_arr_q
) / starting_total_q
grr_quarterly_decomp = (
    starting_total_q - churn_arr_q - contraction_arr_q
) / starting_total_q

decomposition = pd.DataFrame({
    "quarter":       decomp_end_dates.to_period("Q").astype(str),
    "starting_arr":  starting_total_q,
    "new":           new_arr_q,
    "expansion":     expansion_arr_q,
    "contraction":   contraction_arr_q,
    "churn":         churn_arr_q,
    "ending_arr":    ending_total_q,
    "nrr_q":         nrr_quarterly_decomp,
    "grr_q":         grr_quarterly_decomp,
})

money_cols = ["starting_arr", "new", "expansion", "contraction", "churn", "ending_arr"]
pct_cols = ["nrr_q", "grr_q"]
decomposition_display = decomposition.copy()
for col in money_cols:
    decomposition_display[col] = decomposition_display[col].map(lambda v: f"£{v/1_000_000:.2f}m")
for col in pct_cols:
    decomposition_display[col] = (decomposition_display[col] * 100).map(lambda v: f"{v:.1f}%")
decomposition_display

In [ ]:
quarters_q  = decomposition["quarter"].to_numpy()
x_q         = np.arange(len(quarters_q))

new_m        = decomposition["new"].to_numpy()        / 1_000_000
expansion_m  = decomposition["expansion"].to_numpy()  / 1_000_000
contraction_m= decomposition["contraction"].to_numpy()/ 1_000_000
churn_m      = decomposition["churn"].to_numpy()      / 1_000_000
net_m        = new_m + expansion_m - contraction_m - churn_m

fig, ax = plt.subplots(figsize=(10, 6))
ax.axhline(0, color=PALETTE["neutral"], linewidth=0.8, zorder=0)

# Hatch patterns give each movement a distinct texture so greyscale
# readers can separate New, Expansion, Contraction, and Churn without
# relying on colour luminance alone.
ax.bar(x_q, new_m,        color=MOVEMENT_COLOURS["new"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="",    label="New ARR")
ax.bar(x_q, expansion_m,  bottom=new_m, color=MOVEMENT_COLOURS["expansion"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="///", label="Expansion")
ax.bar(x_q, -contraction_m, color=MOVEMENT_COLOURS["contraction"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="...", label="Contraction")
ax.bar(x_q, -churn_m, bottom=-contraction_m, color=MOVEMENT_COLOURS["churn"],
       edgecolor=PALETTE["background"], linewidth=0.4,
       hatch="xxx", label="Churn")

ax.plot(x_q, net_m, marker="o", linewidth=1.6, markersize=6,
        color=PALETTE["primary"], label="Net ARR change")

positive_first  = new_m[0] + expansion_m[0]
positive_last   = new_m[-1] + expansion_m[-1]

ax.set_xticks(x_q)
ax.set_xticklabels(quarters_q, rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("ARR movement (£m)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"£{v:.1f}m"))
ax.set_title(
    f"Positive ARR flows halve from £{positive_first:.1f}m to £{positive_last:.1f}m per quarter; "
    "expansion compression drives the decline",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(loc="upper right", frameon=False, ncol=1, fontsize=9)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "06_nrr_decomposition.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

In [ ]:
ttm_start_total_full = (start_arr_matrix.to_numpy() * cohort_mask).sum(axis=0)
# GRR caps each customer's ending ARR at their Q-4 baseline so
# expansion is removed from the numerator but churn/contraction bite.
ttm_capped = np.minimum(end_arr_matrix.to_numpy(), start_arr_matrix.to_numpy())
ttm_grr_total = (ttm_capped * cohort_mask).sum(axis=0)

grr_ttm = ttm_grr_total / ttm_start_total_full
nrr_ttm = nrr_quarterly["nrr"].to_numpy()
quarters_ttm = nrr_quarterly["quarter"].to_numpy()
x_ttm = np.arange(len(quarters_ttm))

fig, ax = plt.subplots(figsize=(10, 6))
ax.axhline(1.0, linestyle=":", color=PALETTE["neutral"], linewidth=1.0, label="100% reference")

ax.plot(x_ttm, nrr_ttm, marker="o", linewidth=2.2, markersize=7,
        color=PALETTE["primary"], label="NRR (trailing-12-months)")
ax.plot(x_ttm, grr_ttm, marker="s", linewidth=2.2, markersize=7,
        linestyle="--", color=PALETTE["accent"], label="GRR (trailing-12-months)")

grr_low, grr_high = float(grr_ttm.min()) * 100, float(grr_ttm.max()) * 100
nrr_first, nrr_last = float(nrr_ttm[0]) * 100, float(nrr_ttm[-1]) * 100

ax.set_xticks(x_ttm)
ax.set_xticklabels(quarters_ttm, rotation=45, ha="right")
ax.set_xlabel("Quarter")
ax.set_ylabel("Retention (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.set_ylim(0.85, 1.25)
ax.set_title(
    f"GRR holds between {grr_low:.0f}% and {grr_high:.0f}% while NRR falls from "
    f"{nrr_first:.0f}% to {nrr_last:.0f}%: an expansion problem, not a churn problem",
    loc="left", color=PALETTE["text"], fontsize=12,
)
ax.legend(loc="lower left", frameon=False)

add_source_annotation(fig)
fig.tight_layout()
fig.savefig(
    FIGURES_DIR / "06_nrr_vs_grr.png",
    dpi=150, bbox_inches="tight",
)
plt.show()

The decomposition splits the headline decline cleanly. Gross Revenue
Retention — the churn-only retention lens, with expansion removed —
holds in a narrow band across all twelve quarters: its lowest point is
still above 91% and its high point sits near 97%. Across the same
window, Net Revenue Retention falls by roughly eleven percentage
points. The gap between the two lines is expansion contribution, and
that gap is what has compressed: quarterly positive ARR flows (new
plus expansion) run around £2m in 2023 but sit closer to £1.2m by
late 2025. Quarterly losses (churn plus contraction) remain modest in
most quarters and peak just above £1m in Q3 2024 — a single spike that
GRR absorbs without dropping below 92%.

This is an expansion problem, not a churn problem. The retention base
is behaving normally; customers who stay, stay, and very few drop to
zero. What has changed is the pace at which existing customers
purchase more seats or move up the product tier. Section 7 asks
whether this compression is uniform across segments or whether it is
concentrated — that segmentation is the next layer of the
diagnostic.